<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
</head>
<body>
    <div style="display: flex; align-items: center;">
        <div>
            <h1>TD 6 - Parameter Recovery </h1>
            <h2>Understanding human behavior with cognitive models</h2>
            <h3>Master in Cognitive Science</h3>
            <h4>École Normale Supérieure - PSL</h4>
            <p> Valentin Wyart - Lecturer<br>
                Amric Trudel - Practical Sessions (TD)<br>
                Notebook author: <a href="mailto:amric.trudel@ens.psl.eu">amric.trudel@ens.psl.eu</a></p>
        </div>
        <div>
            <img src="images/logo_ens.png" style="height: 70px; margin-left: 10px;" />
        </div>
    </div>
</body>
</html>

# Objective
The goal of this TD is to understand how to recover the parameters of a cognitive model from behavioral data. We will :
- Understand the steps to recover parameters from a model
- Implement a parameter recovery loop
- Visualize the results of the parameter recovery
- Build and visualize the confusion matrix


In [1]:
import pandas as pd
from pybads import BADS, OptimizeResult
from tqdm import trange

from bandit_task import BanditTask, RestlessBanditTask
from tests import test_param_recovery_results
from matplotlib import pyplot as plt
import seaborn as sns
import scipy
import numpy as np

## Define a task
We will start by instantiating a bandit task object that will be used to generate the data that we will use to recover the parameters of our model. The `RestlessBanditTask` class is already defined for you in the `bandit_task.py` file, and already imported in the cell above.

📝Instantiate a `RestlessBanditTask` with the following parameters **(no need to re-write the class!)**:
- initial mean of arm 0: `0.5`,
- initial mean of arm 1: `0.5`,
- standard deviation of the rewards: `0.20`,
- drift of the means: `0.15`,
- number of trials: `40`

Plot the task to visualize the evolution of the rewards over time.

⚠️ Make sure the task is not too easy. Since the generation process is stochastic, the trajectories can vary a lot. Make sure the reward trajectories of the two arms are intertwined, otherwise the task will be too easy and recovery will be poor. Relaunch the cell until you get a task that satisfies you.

In [ ]:
task = ...      # Instantiate the class
task.plot()     # This plots the reward trajectories

## Update the RLModel class
We will reuse the `RLModel` class that was we wrote in TD 4 and 5. No need to re-write previous methods, they are already provided, but you should make sure that you understand what they do before you move forward, since you will need to use them:
- `policy()`
- `update()`
- `simulate()`
- `log_likelihood()`

Parameter recovery will require us to fit the `RLModel` many times to different behavioral datasets. Therefore, we need a compact and practical way to run the parameter estimation code. For this, we will use a new Python trick: **class methods**.

### Implement the `fit()` class method
Let's implement the `fit()` class method, that takes as inputs an array of **actions** and an array of **rewards**, and then return a new instance of the `RLModel` class, initialized with the parameters estimated (or fitted) on the provided behavioral data.

🔧*Technical note:* A **class method** is a method that we called from `RLModel` class itself, not from an instance. We will use it as an alternative constructor to create new instances of `RLModel` when we don't know what $\alpha$ and $\tau$ values to use and instead want to estimate them from behavioral data.

Here is an example of how you will use this class method to fit `RLModel` to behavioral data:
```python
rl_model = RLModel.fit(actions, rewards) # Call the class method to create a new instance
rl_model.simulate(task)    # Call its simulate method
```
To compare, here is how we used to instantiate `RLModel`s with specific parameter values:
```python
rl_model = RLModel(0.8, 0.1) # Call the constructor with values for alpha and tau
rl_model.simulate(task) # Call its simulate method
```

📝Implement the `fit()` method in the `RLModel` class. You can adapt the code that you wrote in the `fit_rl_model()` function in TD 5. If you didn't have time to do it, here are the instructions again:
 1. Write the `cost_function()`, that the optimizer needs to evaluate how bad is any combination of *learning rate* and _temperature_ to explain the data. In other words, it gives the "cost" associated with the parameter it receives as input. For this, you need to:
    -  Instantiate an RLModel with the parameters given as arguments (already done)
    -  Compute the log likelihood of the behavior inputted to the `fit()` classmethod, with the model instance you just created.
    -  The returned cost has to be something we wish to **minimize**, so you need to return the **negative log likelihood**.

2. Complete the settings of the BADS optimizer, each of which is a list with two values, one for each parameter in the following order: \[lr, temp]. I let you think about what settings could be appropriate, and we'll discuss it together. As a reference:
    - `x0` represent the typical value that you would expect for each parameter. It's the starting point of the optimization.
    - `lower_bounds` and `upper_bounds` specify the range of values that will be explored. In our case we have bounds of $[0, 1]$ for both parameters
    - `plausible_lower_bound` and `plausible_upper_bound` are more restrictive than the actual bounds and specify the range where we expect to find most solutions.

In [ ]:
class RLModel:
    def __init__(self, learning_rate: float, temperature: float):
        self.learning_rate = learning_rate
        self.temperature = temperature
        self.q_values = np.array([0.5, 0.5])

    def simulate(self, task):
        self.reset()
        actions = []
        probs = []
        rewards = []
        for trial_rewards in task:
            prob = self.policy()
            action = np.random.binomial(1, prob)
            reward = trial_rewards[action]
            self.update(action, reward)
            actions.append(action)
            probs.append(prob)
            rewards.append(reward)
        return np.array(actions, dtype=int), np.array(probs, dtype=float), np.array(rewards, dtype=float)

    def log_likelihood(self, actions: np.ndarray, rewards: np.ndarray) -> float:
        self.reset()
        loglikelihood = 0
        for action, reward in zip(actions, rewards):
            prob_a1 = self.policy()
            prob_chosen_action = prob_a1 if action == 1 else 1 - prob_a1
            prob_chosen_action = np.max([prob_chosen_action, 1e-6])         ## Little trick to avoid numerical errors
            loglikelihood += np.log(prob_chosen_action)
            self.update(action, reward)
        return loglikelihood

    def policy(self):
        np.seterr(over='ignore')  # Silence the warning for overflow
        prob = 1 / (1 + np.exp(-(self.q_values[1] - self.q_values[0]) / self.temperature))
        np.seterr(over='warn')
        return prob

    def update(self, action: int, reward: float):
        td_error = reward - self.q_values[action]
        self.q_values[action] += self.learning_rate * td_error

    def reset(self):
        self.q_values = np.array([0.5, 0.5])

    def __repr__(self):
        return f'RLModel(lr={self.learning_rate: .3f}, t={self.temperature: .3f})'

    @classmethod
    def fit(cls, actions: np.ndarray[int], rewards: np.ndarray[float], verbose: bool = True):
        
        def cost_function(params):
            """This function returns the 'cost' of using the provided 'params' as model parameters on the task."""
            lr, temp = params       # Extracting the parameters being evaluated
            model = cls(lr, temp)   # Calling the RLModel constructor with the evaluated params

            ...                    ## TO DO: Fill the cost function
            cost = ...
            return cost

        optimizer = BADS(
            fun=cost_function,
            x0=[..., ...],     ## Fill this parametrization [lr, temp]
            lower_bounds=[..., ...],
            upper_bounds=[..., ...],
            plausible_lower_bounds=[..., ...],
            plausible_upper_bounds=[..., ...],
            options={
                'display': 'iter' if verbose else 'off'
            }
        )
        result: OptimizeResult = optimizer.optimize()
        optimal_params: np.ndarray = result.x
        learning_rate, temperature = optimal_params
        return cls(learning_rate, temperature)       ## Return a new instance of the class, initialized with optimal_params

📝 Test your implementation manually:
- Instantiate an `RLModel` with parameters of your choice (we will name it `test_model`)
- Simulate the model on the task you previously created (under the variable `task`)
- Fit a new model to the generated behavior
- Fix the bugs 🪲

In [ ]:
# Instantiate an RLModel with parameters of your choice
test_model = ...
# Generate a behavior trajectory
actions, _, rewards = ...
# Fit a new RLModel to the behavior
fitted_model = ...

# Parameter recovery
## Let's start step by step

📝Instantiate an `RLModel` with parameters of your choice.
- Define the original parameters as a dictionary
- Instantiate the model with these parameters

*Python Note:* The `RLModel(**orig_params)` syntax unpacks the content of the `original_params` dictionary and passes each parameter name and value to the constructor of `RLModel`, as "keyword arguments".

In [ ]:
original_params = {
    'learning_rate': ...,   # Fill this
    'temperature': ...
}
original_model = RLModel(**original_params)  # Unpack the dictionary of parameters

📝 Simulate a trajectory with `original_model` and the `task` we defined at the beginning

In [ ]:
actions, probs, rewards = ...

📝 Fit a new `RLModel` to the behavior that you just generated with your original model. We will name it `recovered_model`.

In [ ]:
recovered_model = ...

📝Get the recovered parameters from the model you just fitted

In [ ]:
recovered_params = {
    'recov_learning_rate': ...,
    'recov_temperature': ...
}

📝Store the **original** and **recovered** parameters of your recovery in a pandas `Series`.

_Python Note_: `Series` are the data structures that Pandas uses to hold a **row** of data in a DataFrame. As opposed to a NumPy `array`, a `Series` has an **index** that can hold a distinct **label** to identify each element in the series. You can create a `Series` with a dictionary containing the data. Here you need to combine the content of the `original_parameters` dict and the `recovered_parameters` dict.

In [ ]:
single_result = pd.Series(...)  # Pass a combined dictionary that contains both sets of parameters
single_result                   # This will display the labeled values of your Series

Make sure your `single_result` Series contains the following items:
- learning_rate
- temperature
- recov_learning_rate
- recov_temperature

💭 Are your recovered parameter values close to the original ones? Try with a few different sets of original parameters to see if it seems to work better (or worse)!

## Parameter sampling
But wait, can we do better than choosing parameter values by hand?  
Sure! For that, we can define **parameter distributions** and sample from them. We can instantiate distributions from the `scipy.stats` library, and then sample from them by using their `.rvs()` method (which stands for *random variables sample*)

📝In the `param_distributions` dictionary, instantiate a `scipy.stats` distribution for each parameter of the model. You can have a look at the [scipy documentation](https://docs.scipy.org/doc/scipy/reference/stats.html):
- `learning_rate`: a uniform distribution between 0 and 1
- `temperature`: an exponential distribution with a scale of 0.1


In [ ]:
param_distributions = {
    'learning_rate': ...,
    'temperature': ...
}

👉 Visualize the density of your parameter functions with the `display_distributions()` function below (just run the cell).

In [ ]:
def display_distributions(distributions):
    plt.figure(figsize=(4,3))
    for param, distr in distributions.items():
        x = np.linspace(0, 1, 100)
        y = distr.pdf(x)
        plt.plot(x, y, label=param)
        plt.fill_between(x, 0, y, alpha=0.3)
    plt.legend()
    plt.xlabel('Parameter value')
    plt.ylabel('Density')
    plt.title('Parameter prior distributions')
    plt.show()

display_distributions(param_distributions)

## Full Parameter Recovery Loop
To recap, so far we have been able to:
- Define a bandit **task** and a **model** for which we want to evaluate parameter recovery
- Define **distributions** from which we can **sample** credible parameter values
- Write practical code to **simulate** behavior on the task using the model
- Write practical code to **recover** parameter estimates from the simulated behavior

Now let's put it all together!! 🚀

📝 Complete the `perform_parameter_recovery()` function, that takes the following inputs:
   - the `task` to evaluate (same as always)
   - the `ModelClass` to evaluate (we just pass `RLModel`)
   - the `parameter_distributions` defined just before
   - the number of loop cycles (sampling + simulation + recovery) to perform: `n_samplings`. It's better to set it to a small number (2) while we develop the code.

The function returns the results as a `DataFrame`, with as many rows as `n_samplings`.

In [ ]:
def perform_parameter_recovery(task, ModelClass, parameter_distributions, n_samplings):
    results = []

    for i in trange(n_samplings):
        # Sample parameters and instantiate the original model
        original_params = {
            ...
        }
        original_model = ...

        # Simulate task with the original model
        actions, probs, rewards = ...

        # Fit the model to the simulated data (Tip: use the verbose=False argument)
        recovered_model = ...

        # Extract the recovered parameters
        recovered_params = {
            ...
        }
        # Store the result in a Pandas Series
        single_result = pd.Series(...)

        results.append(single_result)

    return pd.concat(results, axis=1).transpose()


# We call the function and see the results
dev_results = perform_parameter_recovery(task, RLModel, param_distributions, 2)
dev_results

Test your `results` output DataFrame

In [ ]:
test_param_recovery_results(dev_results)

### Generate your parameter recovery data

⚙️ Once your recovery loop works correctly, you can increase the number of iterations and produce proper results. The number you choose will depend on your computer's power. Ideally you should aim for `n_samples` between 30 and 50, but if you see that it will take more than a few minutes, then decrease the number of samples.

💡 In your future projects, you will probably want to parallelize the loop iterations to speed this up by using all of your CPU's cores. For reference, a good library for this is [joblib](https://joblib.readthedocs.io/en/stable/).

In [ ]:
n_samplings = 30    # Adjust this
results = perform_parameter_recovery(task, RLModel, param_distributions, n_samplings)

## Visualizing parameter recovery results

### Regression plot
#### Warm-up: learning rate only
Before writing a clean function to plot the results for all parameters, let's first focus on the **learning rate** and visualize how its close the recovered values are to the original ones.

📝 Make a regression plot that contains:
- A scatter plot showing the **original** learning rates on the **x axis**, against the **recovered** learning rate on the **y axis**
- A regression line
- A dashed identity line that indicates the ideal regression line
- Title and axis labels
- Make sure ranges are indentical on both axes

_Note_: You can have a look at the `regplot` function from the `Seaborn` library (see the [documentation](https://seaborn.pydata.org/generated/seaborn.regplot.html)).

In [ ]:
# Your code here

#### Display both parameters at once

Now we will encapsulate this code in a function that can take in a `results` DataFrame and produce a figure with as many subplots as parameters to recover.

_Technical Note_: To make a figure that contains many plots, you need to call the `plt.subplots(n_lines, n_columns)` method, which returns a `Figure` (`fig`) and an array of `Axes` (`axs`) (See the [documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html).

The `Figure` object refers to the entire figure, and each `Axes` in the `axs` array contains a different scatter plot. Plotting on specific `Axes` requires a slightly different syntax then when we use `plt` with a signle plot. For example, if you want to create a sns.regplot in very first subplot (index 0), you will have to:
- Extract the corresponding `Axes` like this: `ax = axs[0]`
- Pass this `Axes` to the `ax` keyword argument of the `sns.regplot(..., ax=ax)` function to tell it to create the plot in that specific `Axes`
- Instead of calling `plt.plot()`, you then have to call `ax.plot()`
- Careful, the name of the methods change a bit:
    - `plt.title()` -> `ax.set_title()`
    - `plt.xlabel()` -> `ax.set_xlabel()`

🥺This part of Matplotlib is extremely confusing the first time you see it, so don't hesitate to ask for help if you need. But don't worry, it will be worth mastering this for your future research projects 😌.

📝Fill the `plot_results_scatter()` function below, which takes the `results` DataFrame as input and plots two scatter plots side-by-side: learning rate and temperature.

💪 For those who are more advanced in Python, you can try to make the function more general by having it extract automatically the parameter names from the columns of the `results` DataFrame and determine the number of subplots accordignly. This will eventually be useful if you want to reuse this function for other models with different parameters.

In [ ]:
def plot_results_scatter(results: pd.DataFrame):
    # We create a figure with 2 subplots (1 row and 2 columns)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))

    ### Learning rate
    ax = axs[0] # We select the left Axes
    # Scatter plot for the learning rate
    ...

    # Identity line
    ...

    # Axis labels and title
    ...
    ax.set_title('First title to change')


    ### Temperature
    ax = axs[1] # We select the right Axes
    # Scatter plot for the temperature
    ...

    # Identity line
    ...

    # Axis labels and title
    ...
    ax.set_title('Second title to change')

    plt.show()


# Call your function
plot_results_scatter(results)

### Parameter confusion matrix
To make the confusion matrix, we have to proceed in steps:

📝1) Compute the correlation between all columns of your `results` DataFrame. For this, Pandas DataFrames have a `.corr()` method.

In [ ]:
all_correlations = ...
all_correlations

📝2) From you correlation matrix, extract the **columns** associated with the original parameter values and the **rows** associated with the recovered parameter values. 

This should give you the **parameter confusion matrix** (2x2)

In [ ]:
confusion_matrix = ...
confusion_matrix

📝 3) Pack all of this in the `plot_results_confusion_matrix()` function and add a heatmap visualization.

For the heatmap:
- You can use `seaborn.heatmap()`
- Add annotations with the correlation coefficients in the cells
- Use a colormap that is white at 0 and with a different color in the positive and negative values

In [ ]:
def plot_results_confusion_matrix(result_df: pd.DataFrame):
    # Compute all correlations
    all_correlations = ...

    # Compute the confusion matrix
    confusion_matrix = ...

    # Produce a heatmap
    ...
    
    plt.show()

plot_results_confusion_matrix(results)

## Parameter recovery on an easy task
Let's now see how the recovery happens when the task is too easy.

📝 Instantiate a stationary `BanditTask` with:
- Means 0.8 and 0.2
- Standard deviation of 0.05
- 40 trials

In [ ]:
easy_task = ...
easy_task.plot()

📝 Run the parameter recovery and get the results as a DataFrame that you will name `easy_results`

In [ ]:
easy_results = ...

📝 Plot the results (Scatter)

In [ ]:
...

📝Plot the confusion matrix

In [ ]:
...

💭 Can you explain the difference in recoverability between the two parameters?

### 💪 Optional: Histograms
If you have time, you can add a second row to your scatter visualization figure and add, under each scatter plot, a histogram of the differences between the original and recovered parameter values. This will allow you to assess the **bias** and **variance** of your parameter recovery.

In [ ]:
def plot_results_scatter_and_hist(result_df: pd.DataFrame):
    # Create four subplots (2 rows and 2 columns)
    fig, axs = plt.subplots(2, 2, figsize=(12, 8))

    ...

## 💪💪 Optional: Model with 3 parameters
If you want an extra challenge, you can extend the RLModel class to create a model with a third parameter, the **bias**. The bias represents a preference of the participant for arm 1, and is expressed in the softmax policy (refer to the class slides to find the exact equation).

Perform parameter recovery with this model.

### Implement the `BiasRLModel` class

In [ ]:
class BiasRLModel(RLModel):
    def __init__(self, learning_rate: float, temperature: float, bias: float):
        ...

    def policy(self):
        ...

    # Reimplement any method that needs to be re-implemented

### Perform parameter recovery

In [ ]:
...

### Visualize the results
Make sur your plotting functions are generic and can accomodate an arbitrary number of parameters

In [ ]:
...